<a href="https://colab.research.google.com/github/Princekaga/Iterative-Summarization-Refinement/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch bitsandbytes accelerate datasets pandas
from transformers.pipelines.pt_utils import KeyDataset

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import torch
from transformers import pipeline, BitsAndBytesConfig
from datasets import Dataset

Exception ignored in: <function Group.__del__ at 0x7c9eba2b25c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/regex/_regex_core.py", line 3127, in __del__
    def __del__(self):

KeyboardInterrupt: 


In [ ]:
csv_filename = "customer_reviews.csv"

# If the file doesn't exist, we create a mock dataset automatically
if not os.path.exists(csv_filename):
    print(f"'{csv_filename}' not found. Creating a sample dataset for you...")
    mock_data = {
        "review_text": [
            "I absolutely love this new phone, the battery life easily lasts two days!",
            "This was a terrible experience. The food arrived completely cold.",
            "The movie had great acting, but the plot was incredibly boring and slow.",
            "The product arrived on time. It works exactly as advertised, nothing special."
        ]
    }
    pd.DataFrame(mock_data).to_csv(csv_filename, index=False)

# Load the CSV file into a Pandas DataFrame
print(f"Loading {csv_filename} into Pandas...")
df = pd.read_csv(csv_filename)

# Convert the Pandas DataFrame into a Hugging Face Dataset for high-speed processing
hg_dataset = Dataset.from_pandas(df)

'customer_reviews.csv' not found. Creating a sample dataset for you...
Loading customer_reviews.csv into Pandas...


In [ ]:
print("Configuring 4-bit compression settings...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

Configuring 4-bit compression settings...


In [ ]:
print("Loading the Mistral-7B-Instruct model into GPU memory...")
analyzer = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.2",
    model_kwargs={"quantization_config": quantization_config},
    device_map="auto"
)
analyzer.tokenizer.pad_token_id = analyzer.model.config.eos_token_id

Loading the Mistral-7B-Instruct model into GPU memory...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
def format_instruction(batch):
    prompts = []
    for text in batch["review_text"]:
        prompts.append(
            f"[INST] You are a sentiment analyzer. Read this review: '{text}'. "
            f"Reply with exactly one word from these options: POSITIVE, NEGATIVE, MIXED, or NEUTRAL. [/INST]"
        )
    return {"formatted_prompt": prompts}

# Apply the formatting across the entire dataset in fast batches
hg_dataset = hg_dataset.map(format_instruction, batched=True)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [ ]:
print("Starting high-speed batch sentiment analysis...")

results = []
# batch_size=4 processes multiple rows at the exact same time on the GPU
# Adjust batch_size higher (e.g., 8, 16, 32) if you have a larger GPU
for output in analyzer(KeyDataset(hg_dataset, "formatted_prompt"), batch_size=8, max_new_tokens=5):
    # Extract just the generated text that comes AFTER the closing [/INST] tag
    raw_answer = output[0]['generated_text'].split('[/INST]')[-1].strip()
    results.append(raw_answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting high-speed batch sentiment analysis...


In [ ]:
df["predicted_sentiment"] = results

output_filename = "analyzed_customer_reviews.csv"
df.to_csv(output_filename, index=False)

print("\n==================================================")
print(f"SUCCESS! Results successfully saved to '{output_filename}'")
print("==================================================\n")

# Display a preview of the final dataset
print(df[["review_text", "predicted_sentiment"]])


SUCCESS! Results successfully saved to 'analyzed_customer_reviews.csv'

                                         review_text predicted_sentiment
0  I absolutely love this new phone, the battery ...           POSITIVE.
1  This was a terrible experience. The food arriv...           Negative.
2  The movie had great acting, but the plot was i...           NEGATIVE.
3  The product arrived on time. It works exactly ...            NEUTRAL.
